# Notebook 4: Topic Modeling (LDA)

Apply Latent Dirichlet Allocation to article abstracts (1988–2020) to identify the 21 topics in academic finance, replicating the paper's approach with Gensim and spaCy.

**Key finding to verify:** No topic relating to group-based inequality or disadvantage emerges.

**Output:** `data/processed/articles_with_topics.parquet`

In [ ]:
import pandas as pd
import numpy as np
import spacy
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore, CoherenceModel
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import os
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Load spaCy model
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

os.makedirs("figures", exist_ok=True)
print("Libraries loaded.")

In [ ]:
# Load articles and filter to those with abstracts (1988-2020)
articles_df = pd.read_parquet("data/processed/articles_gendered.parquet")

lda_articles = articles_df[
    (articles_df["abstract"].notna()) &
    (articles_df["abstract"].str.len() > 50) &
    (articles_df["publication_year"] >= 1988) &
    (articles_df["publication_year"] <= 2020)
].copy().reset_index(drop=True)

print(f"Articles with abstracts (1988-2020): {len(lda_articles):,}")
print(f"Paper reference: 55,210 abstracts")

## 1. Text Preprocessing with spaCy

Lemmatize, remove stopwords, punctuation, numbers, and short tokens.

In [ ]:
def preprocess_text(text: str) -> list[str]:
    """Preprocess a single abstract for LDA."""
    doc = nlp(text)
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.like_num
        and len(token.text) > 2
        and token.is_alpha
    ]
    return tokens


# Process all abstracts using spaCy's pipe for efficiency
print("Preprocessing abstracts (this may take 20-40 minutes)...")

abstracts = lda_articles["abstract"].tolist()
processed_docs = []

# Process in batches using nlp.pipe
batch_size = 1000
for i in tqdm(range(0, len(abstracts), batch_size), desc="Preprocessing"):
    batch = abstracts[i : i + batch_size]
    for doc in nlp.pipe(batch, batch_size=256):
        tokens = [
            token.lemma_.lower()
            for token in doc
            if not token.is_stop
            and not token.is_punct
            and not token.like_num
            and len(token.text) > 2
            and token.is_alpha
        ]
        processed_docs.append(tokens)

print(f"\nProcessed {len(processed_docs):,} documents")
print(f"Average tokens per document: {np.mean([len(d) for d in processed_docs]):.0f}")

## 2. Build Dictionary and Corpus

In [ ]:
# Build Gensim dictionary
dictionary = Dictionary(processed_docs)
print(f"Dictionary size before filtering: {len(dictionary):,}")

# Filter extremes: remove words in <20 docs or >50% of docs
dictionary.filter_extremes(no_below=20, no_above=0.5)
print(f"Dictionary size after filtering: {len(dictionary):,}")

# Create bag-of-words corpus
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]
print(f"Corpus size: {len(corpus):,} documents")

## 3. Train LDA Model (K=21 topics)

The paper identified 21 topics. We train with K=21 to replicate, and also compute coherence scores for a range of K values to validate.

In [ ]:
# Train LDA with K=21 topics
print("Training LDA model with 21 topics...")
lda_model = LdaMulticore(
    corpus=corpus,
    id2word=dictionary,
    num_topics=21,
    random_state=42,
    passes=10,
    workers=3,
    chunksize=2000,
    alpha="asymmetric",
    eta="auto",
)

# Compute coherence
coherence_model = CoherenceModel(
    model=lda_model,
    texts=processed_docs,
    dictionary=dictionary,
    coherence="c_v",
)
coherence_score = coherence_model.get_coherence()
print(f"\nCoherence score (c_v) for K=21: {coherence_score:.4f}")

# Save model
model_dir = "data/processed/lda_model"
os.makedirs(model_dir, exist_ok=True)
lda_model.save(os.path.join(model_dir, "lda_21topics"))
dictionary.save(os.path.join(model_dir, "dictionary"))
print(f"Model saved to {model_dir}/")

In [ ]:
# Coherence score sweep to validate K=21
print("Computing coherence for different K values (this takes a while)...")
k_values = list(range(5, 41, 5))
coherence_scores = []

for k in tqdm(k_values, desc="K sweep"):
    model_k = LdaMulticore(
        corpus=corpus, id2word=dictionary, num_topics=k,
        random_state=42, passes=5, workers=3, chunksize=2000,
    )
    cm = CoherenceModel(model=model_k, texts=processed_docs, dictionary=dictionary, coherence="c_v")
    coherence_scores.append(cm.get_coherence())

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_values, coherence_scores, "bo-")
ax.axvline(x=21, color="red", linestyle="--", label="K=21 (paper)")
ax.set_xlabel("Number of Topics (K)")
ax.set_ylabel("Coherence Score (c_v)")
ax.set_title("Topic Coherence by Number of Topics")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/coherence_scores.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nCoherence scores:")
for k, c in zip(k_values, coherence_scores):
    marker = " <-- paper" if k == 20 else ""
    print(f"  K={k:3d}: {c:.4f}{marker}")

## 4. Examine Topics and Check for Inequality Topic

In [ ]:
# Display top words for each topic
print("=" * 80)
print("TOP 15 WORDS PER TOPIC")
print("=" * 80)

inequality_terms = {
    "gender", "sex", "female", "male", "woman", "women", "girl", "boy",
    "discrimination", "inequality", "diversity", "bias", "minority",
    "race", "racial", "ethnic", "disadvantage", "marginali", "exclusion",
    "stereotype", "prejudice",
}

has_inequality_topic = False

for topic_id in range(lda_model.num_topics):
    top_words = lda_model.show_topic(topic_id, topn=15)
    words_str = ", ".join([f"{word} ({prob:.3f})" for word, prob in top_words])
    
    # Check for inequality-related terms
    topic_words = {w for w, _ in top_words}
    overlap = topic_words & inequality_terms
    flag = " *** INEQUALITY-RELATED ***" if overlap else ""
    if overlap:
        has_inequality_topic = True
    
    print(f"\nTopic {topic_id:2d}{flag}:")
    print(f"  {words_str}")

print("\n" + "=" * 80)
if has_inequality_topic:
    print("WARNING: Found topic(s) with inequality-related terms.")
else:
    print("CONFIRMED: No topic relating to group-based inequality was identified.")
    print("This matches the paper's key finding.")
print("=" * 80)

In [ ]:
# Generate word clouds for all topics
fig, axes = plt.subplots(3, 7, figsize=(28, 12))
axes = axes.flatten()

for topic_id in range(21):
    topic_words = dict(lda_model.show_topic(topic_id, topn=50))
    wc = WordCloud(
        width=400, height=300, background_color="white",
        colormap="viridis", max_words=50,
    ).generate_from_frequencies(topic_words)
    
    axes[topic_id].imshow(wc, interpolation="bilinear")
    axes[topic_id].set_title(f"Topic {topic_id}", fontsize=10)
    axes[topic_id].axis("off")

plt.suptitle("Word Clouds for 21 LDA Topics in Academic Finance", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("figures/topic_wordclouds.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Assign Dominant Topic to Each Article and Save

In [ ]:
# Assign dominant topic to each article
dominant_topics = []
topic_distributions = []

for i, bow in enumerate(tqdm(corpus, desc="Assigning topics")):
    topic_dist = lda_model.get_document_topics(bow, minimum_probability=0.0)
    # Sort by probability to get dominant topic
    topic_dist_sorted = sorted(topic_dist, key=lambda x: x[1], reverse=True)
    dominant_topic = topic_dist_sorted[0][0]
    dominant_prob = topic_dist_sorted[0][1]
    
    dominant_topics.append({
        "dominant_topic": dominant_topic,
        "topic_probability": dominant_prob,
    })
    
    # Store full distribution for later analysis
    topic_distributions.append([prob for _, prob in sorted(topic_dist)])

# Add to dataframe
lda_articles["dominant_topic"] = [d["dominant_topic"] for d in dominant_topics]
lda_articles["topic_probability"] = [d["topic_probability"] for d in dominant_topics]

# Save
lda_articles.to_parquet("data/processed/articles_with_topics.parquet", index=False)

print(f"\nSaved articles with topics: {len(lda_articles):,}")
print(f"\nTopic distribution:")
print(lda_articles["dominant_topic"].value_counts().sort_index())